# Linear Regression
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
Regression models are commonly used models in data science. They are very explainable so you not only build a model, but you also learn more about relationships in the data. Linear regressions predict a continuous target variable.


# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data

This version of the Toyota Corolla data has already been preprocessed and is ready for modeling.  

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 5/ToyotaCorolla1000.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 1000 rows and 10 columns
df.shape

In [ ]:
# Preview
print(df.head())

We know from prior exploration that fuel type has three values - CNG, Diesel, and Petrol.  So we need dummy variables before building a regression model.

In [ ]:
# Create dummy variables for 'Fuel Type'
df = pd.get_dummies(df, columns=['Fuel Type'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Now let's partition the data to get it ready for modeling. We define Price as the target variable and the rest of the columns as our predictor variables.  We will do a 50/30/20 split.  In the code we first separate out the 20% for test, then we split the remaining portion into training and validation.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Price'
features = df.drop(target, axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

# Linear Regression - Manual Math

Let's start with a simple example using just Mileage to predict Price.  This will allow us to do the calculations manually so you really understand the core concept of a regression model.  

In [ ]:
# Plot Price versus Mileage
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_train['Mileage'], y=y_train)
plt.title('Price vs. Mileage (Training Data)')
plt.xlabel('Mileage')
plt.ylabel('Price')
plt.grid(True)
plt.show()

In [ ]:
# Manual Calculation of Slope (m) and Intercept (b) for Simple Linear Regression
# using only 'Mileage' as the feature and 'Price' as the target

# Extract the 'Mileage' column from the training features and the 'Price' column from the training target
X_train_mileage = X_train['Mileage']
y_train_price = y_train

# Calculate the means of X and y
mean_x = np.mean(X_train_mileage)
mean_y = np.mean(y_train_price)

# Calculate the slope (m)
# m = sum((xi - mean_x) * (yi - mean_y)) / sum((xi - mean_x)^2)
numerator = np.sum((X_train_mileage - mean_x) * (y_train_price - mean_y))
denominator = np.sum((X_train_mileage - mean_x)**2)
m = numerator / denominator

# Calculate the intercept (b)
# b = mean_y - m * mean_x
b = mean_y - m * mean_x

print(f"Calculated Slope (m): {m}")
print(f"Calculated Intercept (b): {b}")

This line is fit to minimize the errors.  How do we define errors?  Each actual point sits some distance away from the line.  We measure how far away each point is using the y axis as the ruler.  The line of best fit has the least error in the in the predictions of y.  

If you want to learn more about fitting the line, watch this great video from Josh Starmer.  https://www.youtube.com/watch?v=PaFPbb66DxQ

Let's look at the line in the data space by building another scatterplot.

In [ ]:
# Plot Price versus Mileage with the manually calculated regression line
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_train['Mileage'], y=y_train)

# Add the regression line
# y = m*x + b
x_values = np.array(X_train['Mileage']) # Use Mileage values from the training data
y_values = m * x_values + b
plt.plot(x_values, y_values, color='red', label=f'Regression Line (y = {m:.2f}x + {b:.2f})')


plt.title('Price vs. Mileage with Regression Line (Training Data)')
plt.xlabel('Mileage')
plt.ylabel('Price')
plt.grid(True)
plt.legend()
plt.show()

This red line is the predictive model built using mileage as a single predictor of price.  

# Linear Regression

##Train the Model

Now lets build a full model using all of our possible predictor variables.  We will train the regression model on the 10 predictors.  8 of these are continuous.  And we have fuel type represented in 2 columns of dummy variables.  CNG is still represented in the data.  It becomes the reference level to which the other categories are compared.  But we will look at that later...

In [ ]:
# Linear Regression with Multiple Predictors

# Initialize the Linear Regression model
model = LinearRegression()

# Fit the model to the training data
model.fit(X_train, y_train)

# Make predictions on the training, validation, and test sets
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Evaluate the model on the training, validation, and test sets
train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Calculate RMSE
train_rmse = np.sqrt(train_mse)
val_rmse = np.sqrt(val_mse)
test_rmse = np.sqrt(test_mse)

train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print the performance metrics
print("Model Performance:")
print(f"  Training Set - RMSE: {train_rmse:.2f}, R-squared: {train_r2:.2f}")
print(f"  Validation Set - RMSE: {val_rmse:.2f}, R-squared: {val_r2:.2f}")
print(f"  Test Set - RMSE: {test_rmse:.2f}, R-squared: {test_r2:.2f}")


# Print the model coefficients and intercept
print("\nModel Coefficients:")
for feature, coef in zip(X_train.columns, model.coef_):
    print(f"  {feature}: {coef:.2f}")

print(f"\nIntercept: {model.intercept_:.2f}")

Performance looks good on training and validation with both RMSE and R squared being similar.  In fact, it actually got a bit better in validation with slightly lower RMSE and higher R squared.  Performance drops quite a bit for test so we will try to figure out why in a bit.

Looking at the coefficients, we get direct clear info about how each predictor impacts the price.  For every extra month that you own a Toyota Corolla, the price is expected to drop by $128.84.  For every extra mile that you drive it, the price is expected to drop by 2 cents.  But things like horse power and weight increase the value of car.  You can see that from the positive coefficients.

When considering the coefficients for the fuel types, the values that you see are in comparison to the category that was dropped when we formed the n-1 dummy variables.  In this case, CNG was dropped.  So cars that run on diesel will be priced \$1644.75 higher than those that run on CNG.  Cars that run on petrol will be \$1017.67 higher than CNG.  and cars that run on diesel will be \$627.08 higher than pertrol.  

Each of those coefficients become a piece of the regression formula below.

In [ ]:
# Print the regression formula

# Get the intercept and coefficients
intercept = model.intercept_
coefficients = model.coef_
feature_names = X_train.columns

# Start building the formula string
formula = f"Price = {intercept:.2f}"

# Add each coefficient and feature name to the formula string
for feature, coef in zip(feature_names, coefficients):
    if coef >= 0:
        formula += f" + {coef:.2f} * {feature}"
    else:
        formula += f" - {-coef:.2f} * {feature}"

# Print the complete formula
print("Regression Formula:")
print(formula)

##Model Performance

In addition to the RMSE and R squared that we already looked at, let's also look at the residuals.  These will help us figure out why the test performance got so much worse.

In [ ]:
# Calculate residuals for each partition
train_residuals = y_train - y_train_pred
val_residuals = y_val - y_val_pred
test_residuals = y_test - y_test_pred

# Plot histograms of the residuals
fig, axs = plt.subplots(1, 3, figsize=(18, 6)) # Increased figure size for better readability

# Training Residuals
sns.histplot(train_residuals, kde=True, ax=axs[0])
axs[0].set_title('Training Residuals Histogram')
axs[0].set_xlabel('Residuals')
axs[0].set_ylabel('Frequency')
train_stats = pd.DataFrame(train_residuals).describe().round(2) # Round to 2 decimal places
axs[0].text(0.05, 0.95, train_stats.to_string(), transform=axs[0].transAxes, va='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

# Validation Residuals
sns.histplot(val_residuals, kde=True, ax=axs[1])
axs[1].set_title('Validation Residuals Histogram')
axs[1].set_xlabel('Residuals')
axs[1].set_ylabel('Frequency')
val_stats = pd.DataFrame(val_residuals).describe().round(2) # Round to 2 decimal places
axs[1].text(0.05, 0.95, val_stats.to_string(), transform=axs[1].transAxes, va='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

# Test Residuals
sns.histplot(test_residuals, kde=True, ax=axs[2])
axs[2].set_title('Test Residuals Histogram')
axs[2].set_xlabel('Residuals')
axs[2].set_ylabel('Frequency')
test_stats = pd.DataFrame(test_residuals).describe().round(2) # Round to 2 decimal places
axs[2].text(0.05, 0.95, test_stats.to_string(), transform=axs[2].transAxes, va='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))


plt.tight_layout()
plt.show()

In the test partition we have at least one car with a residual of -12000 which is much more extreme than in training or validation.  That means that the model is predicting much too high for that car.  That is enough to skew our error calulations for test.  

In [ ]:
# Create dataframes with actual and predicted prices
# This is to ensure the plotting code has the necessary dataframes
if 'train_results' not in locals() or 'val_results' not in locals() or 'test_results' not in locals():
    train_results = X_train.copy()
    train_results['Actual_Price'] = y_train
    train_results['Predicted_Price'] = y_train_pred

    val_results = X_val.copy()
    val_results['Actual_Price'] = y_val
    val_results['Predicted_Price'] = y_val_pred

    test_results = X_test.copy()
    test_results['Actual_Price'] = y_test
    test_results['Predicted_Price'] = y_test_pred

    print("Dataframes with actual and predicted prices created.")

# Plot actual vs. predicted prices for each partition
fig, axs = plt.subplots(1, 3, figsize=(18, 6)) # Increased figure size for better readability

# Training Partition
sns.scatterplot(x='Actual_Price', y='Predicted_Price', data=train_results, ax=axs[0])
axs[0].set_title('Training: Actual vs. Predicted Price')
axs[0].set_xlabel('Actual Price')
axs[0].set_ylabel('Predicted Price')
axs[0].plot([train_results['Actual_Price'].min(), train_results['Actual_Price'].max()],
           [train_results['Actual_Price'].min(), train_results['Actual_Price'].max()],
           color='red', linestyle='--') # Add a diagonal line for reference

# Validation Partition
sns.scatterplot(x='Actual_Price', y='Predicted_Price', data=val_results, ax=axs[1])
axs[1].set_title('Validation: Actual vs. Predicted Price')
axs[1].set_xlabel('Actual Price')
axs[1].set_ylabel('Predicted Price')
axs[1].plot([val_results['Actual_Price'].min(), val_results['Actual_Price'].max()],
           [val_results['Actual_Price'].min(), val_results['Actual_Price'].max()],
           color='red', linestyle='--') # Add a diagonal line for reference

# Test Partition
sns.scatterplot(x='Actual_Price', y='Predicted_Price', data=test_results, ax=axs[2])
axs[2].set_title('Test: Actual vs. Predicted Price')
axs[2].set_xlabel('Actual Price')
axs[2].set_ylabel('Predicted Price')
axs[2].plot([test_results['Actual_Price'].min(), test_results['Actual_Price'].max()],
           [test_results['Actual_Price'].min(), test_results['Actual_Price'].max()],
           color='red', linestyle='--') # Add a diagonal line for reference


plt.tight_layout()
plt.show()

In these scatterplots, we want the points to fall as close to the diagonal as possible.  When a point is exactly on the red line that means that the actual and predicted price are the same with zero error.  Points above the line mean that the prediction was too high.  Points below the line mean the prediction was too low.  

Looking again at Test to see why the model perforemd worse there, we can see a couple fo points that are very distand from the diagonal.  For the worst prediction, there is a car that is actually priced around \$12,000 but the model is predicting its price to be nearly \$25,000!  Can you find which car it is?  Maybe there's something unusal about it.  It would be worth exploring more and then possibly re-running the model without that car if it is an anomaly.  Give it a try on your own!

##Variable Selection

P values can help us understand which variables are helpful in forming a prediction and which ones are not.  The smaller the P value, the better the predictor!

In [ ]:
# Fit the model using statsmodels to get p-values and other statistics

# Convert boolean columns to integers for statsmodels compatibility
X_train_sm = X_train.copy() # Create a copy to avoid modifying the original X_train
for col in ['Fuel Type_Diesel', 'Fuel Type_Petrol']:
    if col in X_train_sm.columns and X_train_sm[col].dtype == bool:
        X_train_sm[col] = X_train_sm[col].astype(int)

# Add a constant to the features for the intercept
X_train_sm = sm.add_constant(X_train_sm)


# Initialize and fit the OLS model
model_sm = sm.OLS(y_train, X_train_sm).fit()

# Print the summary of the regression results
print(model_sm.summary())

There's a lot of output here, but let's focus on the column labeled P>|t| because that gives us the p values of our predictor variables.  Using an alpha of 0.05, there is opportunity to improve the model by dropping the predictors that are not statistically significant.  Can you rerun the regression model without Metalic, Automatic, Doors, and Fuel Type?

We can also automate our variable selection with methods like backward elimination, forward selection, or a mixed approach.  Below is an example of backward elimination which came to the same conclusion as our manual approach above.  

In [ ]:
# Automate variable selection using Backward Elimination based on p-values

def backward_elimination(X, y, significance_level=0.05):
    """
    Performs backward elimination to select features based on p-values.

    Args:
        X (pd.DataFrame): DataFrame of predictor variables.
        y (pd.Series): Series of the target variable.
        significance_level (float): The alpha level for statistical significance.

    Returns:
        list: A list of the selected feature names.
    """
    features = X.columns.tolist()
    while (len(features) > 0):
        # Create a copy and convert boolean columns to integers for statsmodels compatibility
        X_be = X[features].copy()
        for col in X_be.columns:
            if X_be[col].dtype == bool:
                X_be[col] = X_be[col].astype(int)

        X_train_be = sm.add_constant(X_be)
        model_be = sm.OLS(y, X_train_be).fit()
        p_values = model_be.pvalues[1:] # Exclude constant's p-value
        max_p_value = max(p_values)
        if max_p_value > significance_level:
            redundant_feature = p_values.idxmax()
            features.remove(redundant_feature)
        else:
            break
    return features

# Perform backward elimination on the training data
selected_features = backward_elimination(X_train, y_train)

print("Selected features after backward elimination:", selected_features)

# You can now use these selected features to train a new model
X_train_selected = X_train[selected_features]
X_val_selected = X_val[selected_features]
X_test_selected = X_test[selected_features]

# Display the shapes of the selected feature sets
print("\nShape of training data with selected features:", X_train_selected.shape)
print("Shape of validation data with selected features:", X_val_selected.shape)
print("Shape of test data with selected features:", X_test_selected.shape)

So let's run the model again with fewer predictors and see how it performs.

In [ ]:
# Rerun the Linear Regression model with the selected subset of predictors

# Initialize the Linear Regression model
model_selected = LinearRegression()

# Fit the model to the training data with selected features
model_selected.fit(X_train_selected, y_train)

# Make predictions on the training, validation, and test sets with selected features
y_train_pred_selected = model_selected.predict(X_train_selected)
y_val_pred_selected = model_selected.predict(X_val_selected)
y_test_pred_selected = model_selected.predict(X_test_selected)

# Evaluate the new model on the training, validation, and test sets
train_mse_selected = mean_squared_error(y_train, y_train_pred_selected)
val_mse_selected = mean_squared_error(y_val, y_val_pred_selected)
test_mse_selected = mean_squared_error(y_test, y_test_pred_selected)

# Calculate RMSE for the new model
train_rmse_selected = np.sqrt(train_mse_selected)
val_rmse_selected = np.sqrt(val_mse_selected)
test_rmse_selected = np.sqrt(test_mse_selected)

train_r2_selected = r2_score(y_train, y_train_pred_selected)
val_r2_selected = r2_score(y_val, y_val_pred_selected)
test_r2_selected = r2_score(y_test, y_test_pred_selected)

# Print the performance metrics for the new model
print("Model Performance with Selected Features:")
print(f"  Training Set - RMSE: {train_rmse_selected:.2f}, R-squared: {train_r2_selected:.2f}")
print(f"  Validation Set - RMSE: {val_rmse_selected:.2f}, R-squared: {val_r2_selected:.2f}")
print(f"  Test Set - RMSE: {test_rmse_selected:.2f}, R-squared: {test_r2_selected:.2f}")

# Print the model coefficients and intercept for the new model
print("\nModel Coefficients with Selected Features:")
for feature, coef in zip(X_train_selected.columns, model_selected.coef_):
    print(f"  {feature}: {coef:.2f}")

print(f"\nIntercept: {model_selected.intercept_:.2f}")

Performance stayed nearly the same but now we only need to collect, clean, store, and use five variables instead of teh full list that we started with.  That's much more efficient!